# 📡 FreeAlphaRadar — Colab / Kaggle launcher

Zero-config: **no API keys, no sign-ups, no secrets**. This notebook clones the repo, installs the pinned dependencies, seeds the offline sample cache, and launches the Streamlit dashboard via a public tunnel.

- Runs **fully offline** if the runtime can't reach the data sources — it serves the seeded sample data, so the dashboard is never blank.
- With network, click **Refresh Data & Re-score** in the app to pull live data (yfinance via a browser-impersonating `curl_cffi` session with a Stooq fallback, SEC EDGAR, PatentsView, GDELT — all free, no keys).
- An optional final section runs the **market-wide auto-discovery** job (scan all ~8,000 SEC filers → ranked top-10 under-the-radar names).

## 1. Clone the repository

In [ ]:
import os
REPO_URL = "https://github.com/chizkidd/freealpharadar.git"
if not os.path.exists("freealpharadar"):
    !git clone $REPO_URL
%cd freealpharadar

## 2. Install dependencies (pinned for reproducibility)

In [ ]:
# Core install: slim runtime + test tooling. The app runs great on this alone
# (fast lexicon sentiment, full dashboard). curl_cffi is included for resilient
# yfinance fetches from datacenter/Colab IPs.
!pip install -q -r requirements.txt -r requirements-dev.txt

# OPTIONAL — real pre-trained FinBERT + XGBoost. The app behaves identically
# either way; only the sentiment engine swaps in. Colab ships torch already, so
# install just the NLP extras (avoids re-pinning Colab's torch). Uncomment:
# !pip install -q transformers==4.46.3 xgboost==2.1.4

## 3. Seed the offline sample cache

This guarantees the dashboard is populated even with no network access.

In [ ]:
!python run_scorer.py --seed-sample --no-refresh --no-ml

## 4. Run the offline test-suite (optional sanity check)

In [ ]:
!FAR_OFFLINE=1 python -m pytest -q

## 5. Launch Streamlit through a public tunnel

We start Streamlit in the background and expose it with `localtunnel`. The cell prints a public URL and the tunnel password (which is just the Colab VM's external IP).

In [ ]:
# Install localtunnel (no account required).
!npm install -g localtunnel >/dev/null 2>&1

import subprocess, time
# Start Streamlit in the background (headless, no usage stats).
proc = subprocess.Popen(
    ["streamlit", "run", "streamlit_app.py",
     "--server.port", "8501", "--server.headless", "true"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
time.sleep(8)
print("Tunnel password (your external IP):")
!curl -s https://loca.lt/mytunnelpassword || curl -s ifconfig.me
print("\nOpen the URL printed below and enter the password above:")
!npx --yes localtunnel --port 8501

### Alternative: `streamlit-jupyter` inline

If you prefer to render inside the notebook, install `streamlit-jupyter` and run the app object directly. The tunnel approach above is the most reliable on Colab/Kaggle.

## 6. (Optional) Market-wide auto-discovery

Scan **all ~8,000 SEC filers** and rank the top under-the-radar names. This is the offline two-stage funnel (bulk DuckDB screen → full scoring of the shortlist). It downloads SEC bulk data, so it's slower — `--dry-run` just previews the ranking without modifying `universe.txt`.

In [ ]:
# OPTIONAL — market-wide auto-discovery (needs network; downloads SEC bulk data).
# 1) build the fundamentals warehouse for a few recent years (a few hundred MB),
# 2) screen all ~8,000 filers and print a ranked top-10 shortlist.
# --dry-run previews the ranking WITHOUT modifying universe.txt. Takes a while.
!pip install -q -r requirements-warehouse.txt
!python -m freealpharadar.warehouse build --since 2022
!python -m freealpharadar.discovery run --top 10 --no-ml --dry-run